# **AI in Engineering: Foundations and Deployment Fall 2026 - Homework 2**
##### (50 points)

### Student Name: 

*(Double click a block to edit it)*

---

This homework has **two parts**:

| | What you'll do | Points |
|---|---|---|
| **Part 1: Model Building** | Run and modify code to choose the right kind of AI model for three different questions | 20 |
| **Part 2: Deployment Review** | Write a short analysis of what can go wrong when AI is deployed, using a real case: the 2021 Texas blackouts | 30 |

In Part 1, the code cells walk you through the work in Python. **Run each cell in order** — you don't need to modify anything if it isn't specified by a "Task" label.

**Don't worry if some of the code in this notebook looks unfamiliar!** There are plenty of comments throughout explaining what each part does, so be sure to read them as you go.

**What's different from Homework 1?** In HW1 you trained models step by step. This time, we've written two helper functions that do the training for you. Your job is the part engineers actually get paid for: **deciding what kind of model fits the question, choosing the inputs, and judging whether the results can be trusted.**

# PART 1: Model Building *(20 points)*

## 1.0 The Scenario
---

You've just started as a data scientist at *Meridian Grid Authority*, the organization that runs the electric grid for a region with five major cities. A grid operator has one big job: **electricity supply has to match demand at every moment.** 

Power plants must produce exactly as much electricity as people are using. If demand gets too high and there isn't enough supply, the grid can become unstable, and operators may have to cut power to some neighborhoods to protect the rest.

Your manager sends you this message:

 > *"We've collected years of hourly data on electricity demand, power generation, prices, and weather. Leadership wants to use it to plan better for winter and summer. I have three questions:*
>
> 1. **How much electricity will we need tomorrow, hour by hour?**
> 2. **Can we flag the hours when the grid is likely to be under stress?**
> 3. **What are the different "kinds" of days we operate under?**
>
> *Use the data to give us insights on each one and tell us where to go from here."*

You won't build the models from scratch. Instead, you'll use two helper functions *(explained in Section 1.3)* and focus on **choosing the right type of model, choosing the right inputs, and reading the results carefully.**

We'll use the [Hourly Energy Demand, Generation and Weather dataset](https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather) from Kaggle. It has four years (2015–2018) of real hourly records from a national power grid, plus weather from five cities. We're using it as a stand-in for Meridian's own records.

## 1.1 Getting the Data
---

The cells in this section download the dataset and load it into your code.

In [ ]:
# kagglehub is a library that handles logging into Kaggle and downloading
# dataset files for us, so we don't have to manually download a zip file and
# unzip it ourselves.
import kagglehub

# Download the latest version of the dataset. This returns a local folder path
# where the dataset's files have been saved on this machine.
path = kagglehub.dataset_download("nicholasjhana/energy-consumption-generation-prices-and-weather")

print("Path to dataset files:", path)

In [ ]:
# View what file(s) got downloaded into dataset folder
import os

print(os.listdir(path))

In [ ]:
# Like we've seen before, we'll use pandas to work with our data.
import pandas as pd

# This dataset comes as TWO files: one with energy data, one with weather data.
energy = pd.read_csv(f"{path}/energy_dataset.csv")
weather = pd.read_csv(f"{path}/weather_features.csv")

# .head() shows just the first 5 rows, so we can eyeball the data
energy.head()

In [ ]:
# The weather file has one row per city per hour (5 cities, so it's about 5x longer)
weather.head()

## 1.2 Cleaning the Data
___

The two files are messy in a few ways: the weather file has duplicate rows and a few impossible values (like a wind speed of 133 m/s), the temperatures are in Kelvin, and the two files need to be lined up by time. This is very normal for real-world data!

The next cell cleans everything up and combines the two files into **one table called `df`** with one row per hour. 

**You don't need to understand every line** — just read the comments to get the big picture, then run it.

In [ ]:
import numpy as np

# ---- Step 1: Make the time columns match in both files ----
energy["time"] = pd.to_datetime(energy["time"], utc=True)
weather["time"] = pd.to_datetime(weather["dt_iso"], utc=True)

# ---- Step 2: Clean the weather file ----
weather["city_name"] = weather["city_name"].str.strip()                      # one city name had a stray space
weather = weather.drop_duplicates(subset=["city_name", "time"])              # remove duplicate rows
weather.loc[weather["wind_speed"] > 50, "wind_speed"] = np.nan               # impossible values -> "missing"
weather["temp_c"] = weather["temp"] - 273.15                                 # Kelvin -> Celsius

# Turn "one row per city per hour" into "one row per hour, one temperature column per city"
temps = weather.pivot(index="time", columns="city_name", values="temp_c")
temps.columns = ["temp_" + city.lower() for city in temps.columns]

# Average the other weather measurements across the five cities
other_weather = weather.groupby("time").agg(
    humidity=("humidity", "mean"),
    wind_speed=("wind_speed", "mean"),
    clouds=("clouds_all", "mean"),
    rain=("rain_1h", "mean"),
)

# ---- Step 3: Clean the energy file ----
energy = energy.set_index("time")
energy.columns = [c.replace("generation ", "gen_").replace(" ", "_").replace("/", "_").replace("-", "_") for c in energy.columns]
energy = energy.rename(columns={
    "forecast_solar_day_ahead": "forecast_solar",
    "forecast_wind_onshore_day_ahead": "forecast_wind",
    "total_load_forecast": "load_forecast_operator",
    "total_load_actual": "load_actual",
})
# Two columns are completely empty, so we drop them
energy = energy.drop(columns=["forecast_wind_offshore_eday_ahead", "gen_hydro_pumped_storage_aggregated"])

# ---- Step 4: Combine into one table and fill tiny gaps ----
df = energy.join(temps).join(other_weather)
df = df.interpolate(limit=6)          # fill short gaps (a few missing hours) by drawing a line between neighbors

# Drop columns that never change (for example, a power source that is always 0)
df = df.drop(columns=[c for c in df.columns if df[c].nunique() <= 1])

# ---- Step 5: Create some new columns (this is called "feature engineering") ----
local_time = df.index.tz_convert("Europe/Madrid")
df["hour"] = local_time.hour                       # 0-23
df["day_of_week"] = local_time.dayofweek           # 0 = Monday ... 6 = Sunday
df["month"] = local_time.month                     # 1-12
df["year"] = local_time.year
df["is_weekend"] = (local_time.dayofweek >= 5).astype(int)
df["avg_temp"] = df[[c for c in df.columns if c.startswith("temp_")]].mean(axis=1)   # average temperature across the 5 cities

# Past demand: how much electricity was used at this same hour yesterday, and one week ago?
df["load_24h_ago"] = df["load_actual"].shift(24)
df["load_7d_ago"] = df["load_actual"].shift(168)

df = df.dropna()    # drop the first week (we have no "past demand" for it yet)

print("Final table:", df.shape[0], "rows (hours) and", df.shape[1], "columns")
df.head()

## 1.3 Understanding the Data
---

**Data dictionary** for our cleaned table `df`:

| Column(s) | Description | Units |
|---|---|---|
| `load_actual` | Electricity demand in that hour (**how much power people actually used**) | MW (megawatts) |
| `load_forecast_operator` | The grid operator's own forecast of demand, made the day before | MW |
| `load_24h_ago`, `load_7d_ago` | Demand at the same hour yesterday / one week ago (columns we created) | MW |
| `hour`, `day_of_week`, `month`, `year`, `is_weekend` | Calendar information (hour 0–23, Monday = 0, etc.) | — |
| `temp_madrid`, `temp_barcelona`, `temp_valencia`, `temp_seville`, `temp_bilbao` | Temperature in each of the five cities | °C |
| `avg_temp` | Average of the five city temperatures | °C |
| `humidity`, `wind_speed`, `clouds`, `rain` | Weather averaged across the five cities | %, m/s, % cover, mm |
| `gen_...` columns (`gen_nuclear`, `gen_fossil_gas`, `gen_wind_onshore`, `gen_solar`, etc.) | How much electricity each power source produced in that hour | MW |
| `forecast_solar`, `forecast_wind` | Day-ahead forecasts of solar and wind generation | MW |
| `price_day_ahead`, `price_actual` | Price of electricity (forecast the day before, and actual) | € per MWh |

*(Quick note: the weather columns are measured weather. In real life you'd use weather **forecasts**, which are usually pretty good, so we'll pretend we know the weather ahead of time.)*

In [ ]:
# .describe() gives basic statistics for each column. Here we look at a few key ones.
df[["load_actual", "avg_temp", "humidity", "wind_speed", "price_actual"]].describe()

In [ ]:
import matplotlib.pyplot as plt

# Plot 1: demand over two weeks. Notice the daily "rhythm"
two_weeks = df.loc["2017-01-09":"2017-01-22"]
plt.figure(figsize=(10, 4))
plt.plot(two_weeks.index, two_weeks["load_actual"])
plt.ylabel("Demand (MW)")
plt.title("Hourly electricity demand, two weeks in January 2017")
plt.xticks(rotation=30)
plt.show()

# Plot 2: demand vs. temperature, for every hour in the dataset.
# alpha=0.1 makes each dot see-through so overlapping dots show up darker
plt.figure(figsize=(7, 5))
plt.scatter(df["avg_temp"], df["load_actual"], alpha=0.1)
plt.xlabel("Average temperature (°C)")
plt.ylabel("Demand (MW)")
plt.title("Demand vs. temperature")
plt.show()

#### Terminology Note: Two main families of AI models

- **Supervised learning**: the data includes a column with the "right answer" (the **target**), and the model learns to predict it from the other columns (the **features**). Example: predict `strength` from the mixture recipe in HW1. If the target is a *number*, it's called **regression**. If the target is a *category* (like yes/no), it's called **classification**.

- **Unsupervised learning**: there is **no** "right answer" column. The model looks for patterns or groups in the data on its own. A common example is **clustering**: sorting rows into groups of similar rows.

## *Question 1*  *(2 points)*: 

Look back at your manager's three questions in Section 1.0. For **each one**, say whether it is best handled by **supervised** learning (regression or classification), **unsupervised** learning (clustering), or **neither**, and give a one-sentence reason. If you pick supervised, name the column you'd use as the target.

*Write your answer here*

## 1.4 Meet the Helper Functions
---

Run the cell below. It defines two functions you'll use for the rest of Part 1. You don't need to read the code inside (but you're welcome to!). Here's what they do:

**`train_supervised_model(data, training_features, prediction_feature)`**
- You choose which columns are the inputs (`training_features`) and which column to predict (`prediction_feature`).
- It trains an **XGBoost** model. XGBoost is a popular, powerful, general-purpose model that works well on lots of tables of data. Think of it as a "good default."
- It automatically figures out if you're doing regression (predicting a number) or classification (predicting 0/1) based on the target column.
- It splits the data **by time**: it trains on **2015–2017** and tests on **2018**. This is like real forecasting: the model learns from the past and is tested on a "future" it hasn't seen. 
- It reports how well the model did on the 2018 test data and shows plots.

**`train_unsupervised_model_for_clustering(data_to_cluster, n_clusters)`**
- You choose which columns to group by, and how many groups (`n_clusters`) you want.
- It runs **k-means**, a simple method that groups rows with similar values together.
- It shows a color-coded scatter plot of the groups, plus a table of what each group looks like on average.

In [ ]:
# ===================== HELPER FUNCTIONS: just run this cell =====================
# (You don't need to edit anything here.)

from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import (mean_absolute_error, r2_score, accuracy_score,
                             recall_score, ConfusionMatrixDisplay)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from IPython.display import display


def train_supervised_model(data, training_features, prediction_feature):
    # ---- Safety checks, so you get a friendly message instead of a scary error ----
    if len(training_features) == 0:
        print("Your list of features is empty. Add at least one column name and try again!")
        return None
    missing = [c for c in training_features + [prediction_feature] if c not in data.columns]
    if missing:
        print("These columns aren't in the data (check your spelling):", missing)
        return None
    if prediction_feature in training_features:
        print("You can't use the column you're predicting as one of the inputs!")
        return None
    training_features = [c for c in training_features if c != "year"]   # 'year' is only used to split the data

    # ---- Split by time: train on 2015-2017, test on 2018 ----
    train = data[data["year"] < 2018]
    test = data[data["year"] == 2018]
    X_train, y_train = train[training_features], train[prediction_feature]
    X_test, y_test = test[training_features], test[prediction_feature]

    # ---- Regression (predict a number) or classification (predict 0 or 1)? ----
    is_classification = data[prediction_feature].nunique() <= 2

    if is_classification:
        model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
    else:
        model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)

    # Train the model (this is the .fit() step you saw in HW1)
    model.fit(X_train, y_train)
    # Get predictions on the 2018 test data (the .predict() step you saw in HW1)
    predictions = model.predict(X_test)

    print(f"Trained on {len(train):,} hours (2015-2017). Tested on {len(test):,} hours (2018).")
    print(f"Predicting: {prediction_feature}  |  Using {len(training_features)} features")
    print("-" * 60)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    if is_classification:
        accuracy = accuracy_score(y_test, predictions)
        recall = recall_score(y_test, predictions)
        print(f"Accuracy: {accuracy:.1%}   (share of ALL hours labeled correctly)")
        print(f"Recall:   {recall:.1%}   (of the hours that were REALLY a 1, how many did the model catch?)")
        ConfusionMatrixDisplay.from_predictions(y_test, predictions, ax=axes[0], colorbar=False)
        axes[0].set_title("Confusion matrix (rows = real, columns = predicted)")
    else:
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        print(f"MAE: {mae:,.0f}".replace(",", ".") + "   (on average, predictions are off by this much)")
        print(f"R-squared: {r2:.2f}   (1.0 = perfect, 0.0 = no better than guessing the average)")
        print(f"For scale: the average value in the test data is {y_test.mean():,.0f}, so MAE is about {mae / y_test.mean():.1%} of that.")
        axes[0].scatter(y_test, predictions, alpha=0.3, s=8)
        lims = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
        axes[0].plot(lims, lims, color="red", label="Perfect predictions")
        axes[0].set_xlabel("Actual")
        axes[0].set_ylabel("Predicted")
        axes[0].set_title("Predicted vs. actual (2018 test data)")
        axes[0].legend()

    # Second plot: which features did the model lean on most?
    importance = pd.Series(model.feature_importances_, index=training_features).sort_values().tail(10)
    importance.plot(kind="barh", ax=axes[1])
    axes[1].set_title("Which features mattered most to the model?")
    plt.tight_layout()
    plt.show()
    return model


def train_unsupervised_model_for_clustering(data_to_cluster, n_clusters):
    data = data_to_cluster.dropna()
    if data.shape[1] < 2:
        print("Pick at least two columns to cluster on.")
        return None

    # K-means works with distances, so we first put every column on the same scale
    scaled = StandardScaler().fit_transform(data)

    # Run k-means: it finds n_clusters groups of similar rows. No "right answer" column is used!
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = pd.Series(kmeans.fit_predict(scaled), index=data.index, name="cluster")

    # Color-coded scatter plot of the first two columns you chose
    first, second = data.columns[0], data.columns[1]
    plt.figure(figsize=(8, 5))
    for c in range(n_clusters):
        members = data[labels == c]
        plt.scatter(members[first], members[second], alpha=0.6, label=f"Cluster {c}")
    plt.xlabel(first)
    plt.ylabel(second)
    plt.title(f"k-means with {n_clusters} clusters")
    plt.legend()
    plt.show()
    if data.shape[1] > 2:
        print(f"(The plot only shows the first two columns, but all {data.shape[1]} columns were used to make the groups.)")

    # What does each cluster look like on average?
    summary = data.groupby(labels).mean().round(1)
    summary.insert(0, "rows_in_cluster", labels.value_counts().sort_index())
    display(summary)
    return labels

## 1.5 Manager's Question 1: Forecasting Demand
---

*"How much electricity will we need tomorrow, hour by hour?"*

Demand is a **number** and we have lots of history of it, so this is a **supervised regression** problem: the target is `load_actual`.

Let's start with a simple example. We'll predict demand from just four features: the hour of the day, the day of the week, the month, and the average temperature.

In [ ]:
# These are the inputs (X) for our example model
example_features = ["hour", "day_of_week", "month", "avg_temp"]

# The first thing we pass in is the whole table (df), then our list of input columns,
# then the name of the column we want to predict.
example_model = train_supervised_model(df, example_features, "load_actual")

## *Task 1: Try it (2 points)* 

Can you beat the example? Edit `my_features` in the cell below to choose your own inputs, then run it. Try to get a **lower MAE** than the example.

**A good rule of thumb:** only use columns you would actually **know ahead of time** when making a forecast for tomorrow. For tomorrow's demand, you would know the calendar (hour, day, month), the weather forecast, and *past* demand (like `load_24h_ago` and `load_7d_ago`). Do **not** use `load_actual` itself, `year`, or anything that would only be measured *during* tomorrow.

Here are the columns available to you:

In [ ]:
print(list(df.columns))

In [ ]:
# TODO: choose your own list of features. Example format: ["hour", "avg_temp"]
my_features = []

my_model = train_supervised_model(df, my_features, "load_actual")

Now for a curious experiment. What happens if we let the model use **every** column (except the one we're predicting)? Run the cell below.

In [ ]:
# Every column except the target (load_actual), the helper column "year", and the stress label we'll create later
all_columns = [c for c in df.columns if c not in ["load_actual", "year", "stress_hour"]]

everything_model = train_supervised_model(df, all_columns, "load_actual")

## *Question 2*  *(3 points)*: 

**(a)** Which features did you choose in Task 1, and how did your MAE and R² compare to the example model? *(1 point)*

**(b)** The "use every column" model has an enormously better score. Is that good news? Look at the list of columns it used (hint: think about the `gen_` columns and `load_forecast_operator`) and explain why this score is **too good to be true** for the job of forecasting tomorrow's demand. *(2 points)*

*Write your answer here*

## 1.6 Manager's Question 2: Flagging Stress Hours
---

*"Can we flag the hours when the grid is likely to be under stress?"*

Here's a problem: our dataset has no column called "stress." So first we need to **create our own target**. When demand is very high, the grid has very little spare capacity, so we'll call an hour a **stress hour** if its demand is in the top 5% of all hours.

The new column `stress_hour` is **1** for a stress hour and **0** for every other hour. Since the answer is a category (yes/no), this is a **supervised classification** problem.

In [ ]:
# We'll call an hour a "stress hour" if demand is in the top 5% of all hours.
stress_percentile = 0.95
stress_cutoff = df["load_actual"].quantile(stress_percentile)

# (df["load_actual"] > stress_cutoff) is True/False for each row; .astype(int) turns that into 1/0
df["stress_hour"] = (df["load_actual"] > stress_cutoff).astype(int)

print(f"Cutoff for a stress hour: demand above {stress_cutoff:,.0f} MW")
print(f"Share of all hours that are stress hours: {df['stress_hour'].mean():.1%}")

In [ ]:
# Let's try the same four features from before, now predicting stress_hour.
# The function sees that stress_hour only has 0s and 1s, so it automatically does classification.
example_stress_model = train_supervised_model(df, example_features, "stress_hour")

## *Task 2: Tweak it (2 points)* 

Try to build a model that **catches more of the real stress hours** (a higher **recall**). Edit `my_stress_features` below and run it.

**Important:** don't include `load_actual` as a feature. The stress label is built *directly* from it, so that would be cheating!

In [ ]:
# TODO: choose your own list of features
my_stress_features = []

my_stress_model = train_supervised_model(df, my_stress_features, "stress_hour")

## *Question 3*  *(3 points)*: 

Look at the accuracy, recall, and confusion matrix from your Task 2 model.

**(a)** Imagine a "model" that is lazy and **always predicts 0** ("no stress") for every single hour. Roughly what accuracy would it get? (Hint: look at the share of stress hours printed above.) How does that compare to your model's accuracy, and what does that tell you about using accuracy for this problem? *(2 points)*

**(b)** For a grid operator, which mistake is worse: **missing** a real stress hour, or **raising a false alarm**? Why? *(1 point)*

*Write your answer here*

## 1.7 Manager's Question 3: Types of Days
---

*"What are the different 'kinds' of days we operate under?"*

This question is different. Nobody has told us what the "kinds" of days are, so there is no target column to predict. We want the data to *show us* natural groups. That makes it an **unsupervised clustering** problem.

First, we'll make a new table with **one row per day** instead of one row per hour:

In [ ]:
# Group the hourly table by calendar day and summarize each day
local_dates = df.index.tz_convert("Europe/Madrid").date
daily = df.groupby(local_dates).agg(
    avg_temp=("avg_temp", "mean"),         # average temperature that day
    min_temp=("avg_temp", "min"),          # coldest hour
    max_temp=("avg_temp", "max"),          # hottest hour
    peak_load=("load_actual", "max"),      # highest demand that day
    mean_load=("load_actual", "mean"),     # average demand that day
    min_load=("load_actual", "min"),       # lowest demand that day
    mean_price=("price_actual", "mean"),   # average electricity price
    wind_gen=("gen_wind_onshore", "mean"), # average wind power produced
    solar_gen=("gen_solar", "mean"),       # average solar power produced
    is_weekend=("is_weekend", "max"),      # 1 if weekend, else 0
    month=("month", "first"),
    hours_in_day=("hour", "count"),
)
daily = daily[daily["hours_in_day"] == 24].drop(columns="hours_in_day")   # keep complete days only

print("Number of days:", daily.shape[0])
daily.head()

In [ ]:
# Example: group days using two columns (temperature and peak demand), into 3 clusters.
# We pass in just the columns we want to cluster, and how many groups we want.
example_labels = train_unsupervised_model_for_clustering(daily[["avg_temp", "peak_load"]], 3)

## *Task 3: Try it (2 points)* 

Edit the cell below to cluster the days differently. Choose **your own 2–3 columns** from `daily` and **your own number of clusters**, then run it. (Tip: if you want to see how the groups differ, put the two columns you care about *first*, because the plot shows the first two.)

In [ ]:
# TODO: choose your own columns and number of clusters
my_cluster_columns = []
my_n_clusters = 3

my_labels = train_unsupervised_model_for_clustering(daily[my_cluster_columns], my_n_clusters)

## *Question 4*  *(3 points)*: 

Look at the plot and the table of cluster averages for your Task 3 model.

**(a)** Describe each cluster in plain words, like you would for a coworker (for example, *"cold, high-demand weekdays"*). *(2 points)*

**(b)** Clustering never checks its answer against a "right answer." What does that mean for how much you should trust a cluster? What might you try to see if the groups are meaningful (hint: change the number of clusters or the columns)? *(1 point)*


*Write your answer here*

## *Question 5*  *(3 points)*: 

Write a short reply (**4–6 sentences**) to your manager. For each of their three questions, say what the data and models told you, and **one thing you would not trust the model to do yet.**

*Write your answer here*

---
# PART 2: Deployment Review *(30 points)*

## 2.0 Background: What Happened in Texas?
---

<!-- INSTRUCTOR NOTE: verify all figures in this section (14%, one-third of capacity, 200+ deaths, 26 million, ~90%) against primary sources before releasing. -->

After you send your reply from Question 5, the **VP of Operations** writes back:

> *"Thanks. This is helpful. But we've all seen what happened in Texas. Before we start making decisions based on these models, I need to know we're not about to repeat that. Can you review our work and tell me where we might be at risk?"*

You may not have heard of what happened in Texas, so here's the short version.

**What is ERCOT?** ERCOT (the Electric Reliability Council of Texas) is the organization that runs the power grid for most of Texas, roughly 90% of the state's electricity use and about 26 million people. Its job is the same as Meridian's: keep supply and demand balanced at every moment. If it can't, the whole grid can become unstable and shut down, which can take days or weeks to fully restart. To prevent that, operators can order **rolling blackouts**: cutting power to some neighborhoods on purpose so the rest of the grid survives.

Two facts about the Texas grid matter for this story:
- It is mostly **separate from the rest of the US grid**, so it can't easily import extra power from other states in an emergency.
- Power plants in Texas make money by **selling electricity**. There was little reward for spending extra money to prepare equipment for rare events, like extreme cold ("winterizing").

**Winter Storm Uri (February 2021).** A severe cold wave hit Texas, with temperatures far below what the state normally sees. Two things happened at the same time:

1. **Demand shot up.** Millions of people turned on their heaters. The planners' forecasts had **underestimated peak demand by about 14%.**
2. **Supply collapsed.** Power plants, natural gas wells, and pipelines froze. About **a third of the plants** that planners were counting on to turn on when needed (called *dispatchable* capacity) were offline.

With demand far above the forecast and supply far below the plan, ERCOT had to order rolling blackouts. Millions of homes lost power, some for days, in freezing weather. **More than 200 people died**, and the economic damage was estimated in the tens of billions of dollars or more.

**Why this is a good case for us:** the planning models used to prepare for winter were built largely from data on *past* winters, and past winters hadn't looked like this. On top of that, the different groups involved (grid operators, power plant owners, customers, regulators) were each focused on different goals. This is exactly the kind of mismatch we talked about in last week's activity, **"What should the AI optimize?"**

#### The goal of this analysis

The VP wants answers to three things, and the questions below follow that order:

- **Section 2.1:** *Where did things go wrong in Texas?* (Who wanted what, and what was the real source of the failure?)
- **Section 2.2:** *Do we have the same weaknesses in our own work from Part 1?*
- **Section 2.3:** *What should we do about it, and how will we know it worked?*

Write your answers like an engineer explaining to a smart non-expert: **be specific and give reasons.** A few clear sentences per part is plenty.

## 2.1 Where Did Things Go Wrong in Texas?
---

## *Question 6*  *(4 points)*: 

Think back to last week's stakeholder activity. Pick **three** of the following groups involved in the Texas grid:

- **A.** The grid operator (ERCOT)
- **B.** Residents / electricity customers
- **C.** Power plant and natural gas owners
- **D.** State regulators and lawmakers
- **E.** Vulnerable communities (for example, low-income households, elderly people, people who rely on medical equipment)

For each one, describe **what outcome they cared about most** (or what they were rewarded for), and how that goal may have shaped decisions *before* the storm.

*Write your answer here*

## *Question 7*  *(4 points)*: 

There are three places where the failure could have come from:

- **(1) Data and forecasting:** the models were built on past data that didn't include a storm like Uri.
- **(2) Objectives and incentives:** the system was set up to reward low cost, with little reward for preparing for rare events.
- **(3) How the forecast was used:** planners treated one forecast number as "the answer," without planning for the possibility that it was much too low.

Which of these do you think played the **biggest role**, and which came **second**? Defend your ranking with specific reasons from the background above.

*Write your answer here*

## *Question 8*  *(3 points)*: 

Imagine ERCOT's demand forecast had been **perfect**: it predicted the exact peak demand during the storm. Would the blackouts have been avoided? Explain why or why not. What does your answer say about "make the model more accurate" as the only fix?

*Write your answer here*

## 2.2 Do We Have the Same Weaknesses?
---

Now let's turn the same questions on our own work from Part 1. Our demand-forecasting model also learned from past data. How would it handle weather more extreme than what it has seen?

First, let's look at how extreme the weather in our dataset really is:

In [ ]:
# How wide is the range of temperatures our model has ever seen?
print("Years of data:", df["year"].nunique(), f"({df['year'].min()}-{df['year'].max()})")
print(df["avg_temp"].describe().round(1))

Now a **stress test**. We'll train a new forecasting model using only "ordinary" weather, leaving out the **coldest 5%** and **hottest 5%** of hours. Then we'll test it on exactly those extreme hours it never saw. (Our dataset comes from a mild climate with no storm like Uri, so the most extreme weather we have is a stand-in.)

**Run the cell below. You don't need to change anything.**

In [ ]:
# ---- STRESS TEST: train on ordinary weather, test on extreme weather ----
features = ["hour", "day_of_week", "avg_temp"]

# Find the temperatures that mark the coldest 5% and hottest 5% of all hours
cold_cutoff = df["avg_temp"].quantile(0.05)
hot_cutoff = df["avg_temp"].quantile(0.95)
is_cold = df["avg_temp"] <= cold_cutoff
is_hot = df["avg_temp"] >= hot_cutoff

# Split the ordinary-weather hours: 80% to train on, 20% to test on
ordinary = df[~is_cold & ~is_hot]
train_ordinary = ordinary.sample(frac=0.8, random_state=42)
test_ordinary = ordinary.drop(train_ordinary.index)

# Train on ordinary weather only
stress_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
stress_model.fit(train_ordinary[features], train_ordinary["load_actual"])

# Test on three groups: ordinary weather, the coldest hours, and the hottest hours
results = []
for name, part in [("Ordinary weather", test_ordinary), ("Coldest 5% of hours", df[is_cold]), ("Hottest 5% of hours", df[is_hot])]:
    preds = stress_model.predict(part[features])
    results.append({
        "Test group": name,
        "MAE (MW)": round(mean_absolute_error(part["load_actual"], preds)),
        # Average error = predicted minus actual. NEGATIVE means the model UNDER-predicted demand.
        "Average error (MW)": round((preds - part["load_actual"]).mean()),
    })
display(pd.DataFrame(results))

# ---- Plot: what does the model "believe" about demand at different temperatures? ----
# We look at weekday evenings (8 PM) and let only the temperature change.
temps_to_try = np.linspace(df["avg_temp"].min(), df["avg_temp"].max(), 200)
what_if = pd.DataFrame({"hour": 20, "day_of_week": 1, "avg_temp": temps_to_try})
real_evenings = df[(df["hour"] == 20) & (df["day_of_week"] < 5)]

plt.figure(figsize=(9, 5))
plt.scatter(real_evenings["avg_temp"], real_evenings["load_actual"], alpha=0.3, color="gray", label="Real data (weekdays, 8 PM)")
plt.plot(temps_to_try, stress_model.predict(what_if), color="red", linewidth=2, label="Model's prediction")
plt.axvspan(df["avg_temp"].min(), cold_cutoff, color="blue", alpha=0.1, label="Extreme weather the model never saw")
plt.axvspan(hot_cutoff, df["avg_temp"].max(), color="blue", alpha=0.1)
plt.xlabel("Average temperature (°C)")
plt.ylabel("Demand (MW)")
plt.title("What the model learned from ordinary weather")
plt.legend()
plt.show()

## *Question 9*  *(4 points)*: 

Look at the table and the plot.

**(a)** What happened to the model's error on the extreme-weather hours compared to ordinary weather? On the hottest hours, which direction were the errors (too high or too low), and what does the red line do in the blue zone? *(2 points)*

**(b)** Connect this to the Texas story. Which **data limitations** in our Part 1 dataset (for example: how many years, what kinds of weather, how many regions) make Meridian vulnerable to the same kind of surprise? *(2 points)*

*Write your answer here*

## *Question 10*  *(4 points)*: 

In Part 1 we judged our models using **MAE** and **accuracy**. Both treat every mistake the same way. But in the real world, mistakes have different costs for different people.

For **two** of our three Part 1 models (the demand forecast, the stress-hour flag, or the day clusters), name **one stakeholder who benefits** and **one stakeholder who could be hurt or ignored** if Meridian relies on it. Then suggest **one better way to measure success** than plain MAE or accuracy for one of them (for example, a score that punishes under-predicting demand more than over-predicting).



*Write your answer here*

## *Question 11*  *(3 points)*: 

In Part 1, we defined a "stress hour" using `load_actual`, the electricity people **actually used**. During Winter Storm Uri, millions of customers had **no power at all** for days.

**(a)** In data like ours, what would those blackout hours look like in the demand column? Would that show how much power people really *needed*? What problem could this cause for a model trained on that data? *(2 points)*

**(b)** Name **one other way** the future power grid could be different from the 2015–2018 data our model learned from. *(1 point)*

*Write your answer here*

## 2.3 What Should We Do About It?
---

## *Question 12*  *(4 points)*: 

Meridian is thinking about using the Part 1 demand-forecasting model to help plan how much power to line up for next winter.

**(a)** Would you recommend **deploy**, **don't deploy**, or **deploy with conditions**? What decision(s) at Meridian would depend on the model's output, and what is the real-world risk if the forecast is too low during an extreme cold snap? *(2 points)*

**(b)** Choose **two** improvements from the list below (or propose your own). For each, say which limitation from Questions 9–11 it addresses, and name **one resource** (a dataset, an agency, a report, or an expert group) Meridian could use to carry it out. *(2 points)*

- **A. Get data on extreme events:** collect records from other regions or older years that include severe heat waves and cold snaps.
- **B. Run "what if" scenarios:** plan for demand 10–20% *higher* than the forecast and see what breaks.
- **C. Forecast a range, not one number:** produce "likely low / likely high" estimates and plan for the high end.
- **D. Limit the model's range:** have it warn people when the weather is outside what it has seen before, instead of giving a confident answer.
- **E. Bring in the other stakeholders:** include power plant operators, regulators, and community groups when deciding what the system should optimize.
- **F. Keep watching after launch:** track the model's mistakes over time, and set a trigger for when humans must review or retrain it.

*(Resource ideas to get you started: the U.S. Energy Information Administration publishes hourly electric grid data at eia.gov, NOAA publishes historical weather records at ncei.noaa.gov, and federal agencies and universities published detailed reviews of the February 2021 Texas outages. You can use these or find your own!)*

*Write your answer here*

## *Question 13*  *(4 points)*: 

How would you check that your chosen improvements **actually worked**? Be specific about:

- **what you'd measure**,
- **which part of the data** you'd measure it on (hint: is the overall MAE across all hours the right thing to look at here, or something more targeted? Think back to the stress test above), and
- **what result** would make you say "it worked" versus "it didn't."

*Write your answer here*